# 1. Project Overview

The UCI SECOM (Semiconductor Manufacturing) dataset is a classic, highly imbalanced benchmark dataset used for binary classification and feature selection. It represents a real-world manufacturing scenario where signals from hundreds of sensors are used to predict whether a semiconductor wafer will pass or fail a quality test.


## Project Goal
To build optimised models that will efficiently predict whether a semiconductor wafer will pass or fail a quality test.


## Dataset Overview
- **Dataset Name:** UCI SECOM (Semiconductor Manufacturing) dataset
- **Source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/179/secom) 
- **Total Instances:** 1,567 wafers.
- **Total Features:** 591 attributes, primarily representing sensor measurements and process points.
- **Target Variable:** Binary classification (Pass/Fail).
Fail (+1): 104 instances (approx. 6.6%).Pass (-1): 1,463 instances.

## Core Challenges

- **Extreme Class Imbalance:** With only about 6.6% "fail" cases, traditional models often achieve high accuracy by simply predicting "pass" every time, making recall more important than overall accuracy.
- **High Dimensionality:** Many of the 591 features are redundant, contain noise, or have zero variance (the same value for every entry), requiring robust feature selection.
- **Missing Data:** The dataset contains a significant number of null/NaN values, varying in density across different features, which necessitates careful imputation



- **Notes** The Dataset includes .

# Problem Solving

## Import Required Libraries and Dataset

In [ ]:
# We start with installing the desired requirements
import sys
!{sys.executable} -m pip install -r ../requirements.txt

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [2]:
project_root = Path('..') 

raw_data_path = project_root/'data'/'raw'
processed_data_path = project_root/'data'/'processed'

data = pd.read_csv(raw_data_path/'uci-secom.csv')
data.head(100)

,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2008-04-08 20:32:00,3081.07,2560.99,2180.1556,1822.5073,1.2579,100.0,98.1289,0.1261,1.5048,...,NaN,0.5002,0.0098,0.0027,1.9607,0.0277,0.0318,0.0097,114.7497,-1
96,2008-04-08 20:58:00,2992.40,2467.07,2191.6667,1107.4330,1.3529,100.0,103.4233,0.1206,1.4993,...,192.2985,0.4996,0.0326,0.0065,6.5274,0.0095,0.0184,0.0062,192.2985,1
97,2008-04-08 21:43:00,3049.31,2453.82,2265.1889,1740.3297,1.4715,100.0,100.3889,0.1222,1.5310,...,NaN,0.4998,0.0174,0.0049,3.4900,0.0095,0.0184,0.0062,192.2985,-1
98,2008-04-08 22:51:00,3011.49,2537.90,2183.4556,955.9073,1.1048,100.0,102.6978,0.1223,1.6227,...,NaN,0.4955,0.0110,0.0032,2.2104,0.0095,0.0184,0.0062,192.2985,-1


## Exploratory Data Analysis


We shall be examining the Data Structure to understand how the original dataset is organised. We will, scan through the first few rows, examine the
dataset dimension, data types and the descriptive summary table, then finally we will check for missing values and null values.

In [46]:
df = data.copy()
display(df.head(5))


,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


In [47]:
display(df.info())
display(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 1567 entries, 0 to 1566
Columns: 592 entries, Time to Pass/Fail
dtypes: float64(590), int64(1), str(1)
memory usage: 7.1 MB


None

,0,1,2,3,4,5,6,7,8,9,...,581,582,583,584,585,586,587,588,589,Pass/Fail
count,1561.000000,1560.000000,1553.000000,1553.000000,1553.000000,1553.0,1553.000000,1558.000000,1565.000000,1565.000000,...,618.000000,1566.000000,1566.000000,1566.000000,1566.000000,1566.000000,1566.000000,1566.000000,1566.000000,1567.000000
mean,3014.452896,2495.850231,2200.547318,1396.376627,4.197013,100.0,101.112908,0.121822,1.462862,-0.000841,...,97.934373,0.500096,0.015318,0.003847,3.067826,0.021458,0.016475,0.005283,99.670066,-0.867262
std,73.621787,80.407705,29.513152,441.691640,56.355540,0.0,6.237214,0.008961,0.073897,0.015116,...,87.520966,0.003404,0.017180,0.003720,3.578033,0.012358,0.008808,0.002867,93.891919,0.498010
min,2743.240000,2158.750000,2060.660000,0.000000,0.681500,100.0,82.131100,0.000000,1.191000,-0.053400,...,0.000000,0.477800,0.006000,0.001700,1.197500,-0.016900,0.003200,0.001000,0.000000,-1.000000
25%,2966.260000,2452.247500,2181.044400,1081.875800,1.017700,100.0,97.920000,0.121100,1.411200,-0.010800,...,46.184900,0.497900,0.011600,0.003100,2.306500,0.013425,0.010600,0.003300,44.368600,-1.000000
50%,3011.490000,2499.405000,2201.066700,1285.214400,1.316800,100.0,101.512200,0.122400,1.461600,-0.001300,...,72.288900,0.500200,0.013800,0.003600,2.757650,0.020500,0.014800,0.004600,71.900500,-1.000000
75%,3056.650000,2538.822500,2218.055500,1591.223500,1.525700,100.0,104.586700,0.123800,1.516900,0.008400,...,116.539150,0.502375,0.016500,0.004100,3.295175,0.027600,0.020300,0.006400,114.749700,-1.000000
max,3356.350000,2846.440000,2315.266700,3715.041700,1114.536600,100.0,129.252200,0.128600,1.656400,0.074900,...,737.304800,0.509800,0.476600,0.104500,99.303200,0.102800,0.079900,0.028600,737.304800,1.000000


In [48]:
# Check dataset dimension
pd.DataFrame({"Metric": ["Total Observations (Rows)", "Total Sensors/Features (Columns)"],
                "Count": [df.shape[0], df.shape[1]]
             })

,Metric,Count
0,Total Observations (Rows),1567
1,Total Sensors/Features (Columns),592


In [49]:
# Check datatype representation
display(df.dtypes.value_counts())

float64    590
str          1
int64        1
Name: count, dtype: int64

In [50]:
df.select_dtypes(include=['int64', 'object']).head() # to verify 

C:\Users\CJ\AppData\Local\Temp\ipykernel_5460\3020583216.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=['int64', 'object']).head() # to verify


,Time,Pass/Fail
0,2008-07-19 11:55:00,-1
1,2008-07-19 12:32:00,-1
2,2008-07-19 13:17:00,1
3,2008-07-19 14:43:00,-1
4,2008-07-19 15:22:00,-1


In [51]:
# checking for duplicates
df[df.duplicated()]

,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail


In [60]:
# checking for null values
null_list=df.isna().sum().to_list()
print(null_list)

[0, 6, 7, 14, 14, 14, 14, 14, 9, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 10, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 24, 24, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 7, 6, 6, 6, 7, 7, 7, 6, 6, 6, 6, 6, 6, 6, 24, 24, 24, 24, 24, 24, 24, 24, 1, 12, 0, 0, 0, 51, 51, 6, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 2, 2, 6, 6, 6, 6, 715, 0, 0, 0, 0, 0, 24, 0, 0, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 5, 6, 7, 14, 14, 14, 14, 14, 9, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 10, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 24, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 7, 6, 6, 6, 7, 7, 7, 6, 6, 6, 6, 6, 6, 6, 24, 24, 24, 24, 24, 24, 24, 24, 1, 12, 0, 0, 0, 51, 51, 6, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 2, 2, 6, 6, 6, 6, 715, 0, 0, 0, 0, 0, 24, 0, 0, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 5, 6, 7, 14, 14, 14, 14, 14, 9, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 10, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 24, 24, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 

#### Note:

From the above Data Exploration, it can be deduced that our data has 1567 rows and 592 columns.
The columns are in their accurate datatypes and there are no duplicates.

Also, it is observed that there is a lot of varying number of missing values across multiple columns.
Probably due to sensor failures or unrecorded readings.
Some columns have missing values of over 1000 which is about 63% of the data.

Therefore going forward, we will need to drop those columns that has  more than half of its entries as missing or null as they will only serve as noise to our model.

## Data Pre-processing


We will now clean and prepare the data for modelling by handling the null and missing values including feature scaling, splitting and encoding where necessary.

First of all we will calculate the percentage of missing values for each column so that we can then proceed to drop those with more missing than non-missing values.

In [53]:
# Evaluating the percentages of missing values
missing_pct = ((df.isnull().mean() * 100).round()).sort_values(ascending=False)
missing_pct

293          91.0
158          91.0
157          91.0
292          91.0
85           86.0
             ... 
222           0.0
221           0.0
218           0.0
209           0.0
Pass/Fail     0.0
Length: 592, dtype: float64

In [54]:
# Setting our threshold 
threshold = 50  # 50%. thats half
cols_to_drop = missing_pct[missing_pct > threshold].index
df = df.drop(columns=cols_to_drop)
print('Total No. of Columns dropped: ', len(cols_to_drop))

Total No. of Columns dropped:  28


We can now fill the missing values For the Remaining columns with their mean but first we have to seperate the Non-numerical columns from the Numerical.

In [55]:
x = df.drop(columns= ['Time', 'Pass/Fail']) 
y = df['Pass/Fail']

In [56]:
# filling the missing values For the Remaining columns with their mean
X = x.fillna(x.mean())
X.isnull().sum().sum() # for validation

np.int64(0)

Next we compute the variance to remove near-constant columns.

We compute variance to measure how much the data spreads out within the range of values for a particular feature. It is the primary indicator of whether a feature has any useful information to offer or not.
  
We drop them because they provide little to no signal for the model to learn from.

In [61]:
# Evaluating the variance
display(X.shape)
variances = X.var()
low_var_cols = variances[variances < 0.01].index

X = X.drop(columns=low_var_cols) 
print('Total No. of columns dropped : ', variances[variances < 0.01].count(), '\n\n New Dataset Shape:', X.shape)

(1567, 562)

Total No. of columns dropped :  265 

 New Dataset Shape: (1567, 297)


Next, we compute the Correlation matrix to ensure multiple features do not carry the same information

We shall Drop highly correlated features to Prevent multicollinearity and Reduce redundancy.

In [62]:
corr_matrix = (X).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

X = X.drop(columns=to_drop)
X.shape


(1567, 195)

Based on standard Machine Learning Procedures it is best to swap the target variable labels so that
Class 1 signifies the Fail (The rare event) and Class 0 signifies the Pass (The normal event).

therefore we will change the target label '-1' to '0' 

In [63]:
#  to swap the target labels so that -1 becomes (1) and 1 becomes (0)
y = y.map({-1: 0, 1: 1})

In [64]:
y

0       0
1       0
2       1
3       0
4       0
       ..
1562    0
1563    0
1564    0
1565    0
1566    0
Name: Pass/Fail, Length: 1567, dtype: int64

In [65]:
# Confirming the number of classes
y.value_counts()

Pass/Fail
0    1463
1     104
Name: count, dtype: int64

Number of failure cases is really low at about 7% of the entire dataset.

This confirms that there is a severe Class Imbalance therfore we must take this into consideration while building the models.

In [66]:
# Train-test Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [67]:
# Scaling the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Building the Models

### Logistic Regression Model

In [71]:
logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [72]:
y_pred_lr = logreg.predict(X_test_scaled)
y_proba_lr = logreg.predict_proba(X_test_scaled)[:,1]

print("Logistic Regression")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))
print("ROC AUC:", roc_auc_score(y_test, y_proba_lr))

Logistic Regression
[[232  58]
 [ 13  11]]
              precision    recall  f1-score   support

           0       0.95      0.80      0.87       290
           1       0.16      0.46      0.24        24

    accuracy                           0.77       314
   macro avg       0.55      0.63      0.55       314
weighted avg       0.89      0.77      0.82       314

ROC AUC: 0.6617816091954022


### Random Forest Classifier Model 

In [73]:
rf = RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42)
rf.fit(X_train, y_train) # We do not fit tree based models to scaled data

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]

print("Random Forest")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_proba_rf)) 

Random Forest
[[290   0]
 [ 24   0]]
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       290
           1       0.00      0.00      0.00        24

    accuracy                           0.92       314
   macro avg       0.46      0.50      0.48       314
weighted avg       0.85      0.92      0.89       314

ROC AUC: 0.7288074712643678


C:\Users\CJ\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\CJ\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\CJ\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Support Vector Classifier Model

In [75]:
svc_model = SVC(kernel='rbf', probability=True, class_weight='balanced', max_iter=1000, random_state=42)
svc_model.fit(X_train_scaled, y_train)
y_pred_svc = svc_model.predict(X_test_scaled)
y_proba_svc = svc_model.predict_proba(X_test_scaled)[:,1]

print("SVC")
print(confusion_matrix(y_test, y_pred_svc))
print(classification_report(y_test, y_pred_svc))
print("ROC AUC:", roc_auc_score(y_test, y_proba_svc))

SVC
[[282   8]
 [ 21   3]]
              precision    recall  f1-score   support

           0       0.93      0.97      0.95       290
           1       0.27      0.12      0.17        24

    accuracy                           0.91       314
   macro avg       0.60      0.55      0.56       314
weighted avg       0.88      0.91      0.89       314

ROC AUC: 0.6373563218390805


C:\Users\CJ\miniconda3\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


## Model Evaluation & Report

In [77]:
# Comparing the Models
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'SVC'],
    
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_svc),],
    'Recall (Failure)': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_svc),],
    'ROC AUC': [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_svc),]
})

results.sort_values('Recall (Failure)', ascending=False)


,Model,Accuracy,Recall (Failure),ROC AUC
0,Logistic Regression,0.773885,0.458333,0.661782
2,SVC,0.907643,0.125000,0.637356
1,Random Forest,0.923567,0.000000,0.728807


DataPoints with Test/Fail Records = 1 are the defective wafers and they are rare.

This is very critical as In semiconductor manufacturing, the cost of missing a failure is significantly high. 


The primary objective of this analysis is to identify defective wafers (Class 1) from the dataset. Therefore detecting as many failure cases as possible is critical to preventing downstream risks.  


 Given that failures are rare, the dataset exhibits a significant class imbalance. For this reason metrics such as 'Accuracy score' can be misleading we must focus our evaluation on the 'RECALL' metric which will tell us how many failures we were able to identify out of all the failures recorded. 



The RandomForest and Support Vector Classifier models both have high accuracy scores (> 90%) but couldnt detect the failures 

while the logistic regression model though having a lower accuracy score (77%) was able to detect some failures.

### Key Findings

* Both Random Forest and SVC suffered from majority-class bias. Because failures are rare, these models played safe by predicting "Pass" for everything, resulting in high accuracy but zero utility.
* Although less accurate, the Logistic Regression model was more sensitive to the signals of defective wafers. Its 77% accuracy reflects a trade-off that to correctly flag some failures it raised more false alarms.

This initial analysis captures a fundamental challenge in Imbalanced Classification.  
The models struggle to separate the defective from healthy wafers because the have very little "Fail" examples to learn from.


**IN CONCLUSION:**

In predicting whether a semiconductor wafer will pass or fail the quality test the three trained models performed poorly.

But so far, the logistic Regression model has the best performance with the ability to detect some failures.

**RECOMMENDATION/WAY FORWARD**


* Identify the most important features
  
* To try other sampling methods to even the effects of class imbalance

* Implement advanced models with imbalance-aware training

* To optimise the models using hyperparameter tuning

* Lower the classification threshold 



### Very Important!

In [81]:
# Save a copy of the processed data to a seperate folder ('data'/'processed')
clean_df = pd.concat([X, y], axis=1)
project_root = Path('..') 
output_dir = project_root / 'data' / 'processed'
clean_df.to_csv(os.path.join(output_dir, "cleaned_data.csv"), index=False)